<a href="https://www.kaggle.com/code/ayodejiibrahimlateef/cats-vs-dogs-image-classifier-with-efficientnetb0?scriptVersionId=288388346" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Cats vs Dogs Image Classifier with EfficientNetB0 (TensorFlow/Keras)

<p align="center">
  <img src="https://i.ibb.co/DPkRWnHX/Chat-GPT-Image-Dec-25-2025-02-19-00-PM.jpg" width="100%" />
</p>




In this project, I built an end-to-end image classification pipeline to distinguish **cats vs dogs** using the Kaggle *PetImages* dataset (24,959 images across two classes). I started by validating the dataset structure, inspecting sample images, and setting a reproducible workflow with fixed random seeds. Because the dataset contains a few problematic JPEG files, I made the input pipeline robust by enabling error skipping (`Dataset.ignore_errors()`), and I stabilized training by repeating the datasets and defining explicit `steps_per_epoch`/`validation_steps`.

I trained a **transfer learning** model using **EfficientNetB0** pretrained on ImageNet. The model architecture consisted of the EfficientNetB0 backbone with a lightweight classification head: global average pooling, dropout, and a sigmoid output neuron for binary classification. Training was run on Kaggle GPUs with **mixed precision** enabled for performance. After establishing a strong baseline, I fine-tuned the model by unfreezing the last layers of the backbone with a lower learning rate.

To evaluate performance, I computed confusion matrices, classification reports, and ROC-AUC on the validation split. The model achieved around **99% validation accuracy** with **ROC-AUC ≈ 0.9995**. I then performed **threshold tuning** to find the probability cutoff that maximizes F1-score/accuracy, identifying an optimal threshold of **~0.84**, and demonstrated how changing the threshold shifts the balance between false positives and false negatives. Finally, I saved the trained models (best checkpoint and final version) and created a reusable single-image inference function to classify new images consistently using the chosen threshold.


## Set the dataset path + quick sanity check

In [ ]:
import os, glob, random

BASE = "/kaggle/input/kaggle-cat-vs-dog-dataset/kagglecatsanddogs_3367a/PetImages"
CAT_DIR = os.path.join(BASE, "Cat")
DOG_DIR = os.path.join(BASE, "Dog")

print("Cats:", len(glob.glob(CAT_DIR + "/*.jpg")))
print("Dogs:", len(glob.glob(DOG_DIR + "/*.jpg")))
print("Example cat:", glob.glob(CAT_DIR + "/*.jpg")[:1])
print("Example dog:", glob.glob(DOG_DIR + "/*.jpg")[:1])

## Visualize a few images

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def show_samples(folder, n=6, title=""):
    paths = glob.glob(folder + "/*.jpg")
    random.shuffle(paths)
    paths = paths[:n]

    plt.figure(figsize=(12, 4))
    for i, p in enumerate(paths, 1):
        try:
            img = Image.open(p).convert("RGB")
            plt.subplot(1, n, i)
            plt.imshow(img)
            plt.axis("off")
        except:
            plt.subplot(1, n, i)
            plt.text(0.5, 0.5, "CORRUPT", ha="center")
            plt.axis("off")
    plt.suptitle(title)
    plt.show()

show_samples(CAT_DIR, title="Cats")
show_samples(DOG_DIR, title="Dogs")


## Choose your pipeline (TensorFlow / Keras)

## -Build train/val datasets

In [ ]:
import os, glob, math, random
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# Memory growth (prevents TF from grabbing all VRAM at once)
for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except:
        pass

# Mixed precision (good on T4)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())


## Locate dataset folder (robust path)

In [ ]:
BASE = None
for root, dirs, files in os.walk("/kaggle/input"):
    if os.path.basename(root) == "PetImages" and "Cat" in dirs and "Dog" in dirs:
        BASE = root
        break

print("BASE:", BASE)
assert BASE is not None, "PetImages folder not found. Check dataset is attached."


## Build train/val datasets (from directory)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    BASE,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    BASE,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print("Classes:", class_names)


## Skip corrupted images + optimize pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# Skip corrupted/unreadable images safely (no deprecation warning)
train_ds = train_ds.ignore_errors()
val_ds   = val_ds.ignore_errors()

# Improve throughput
train_ds = train_ds.shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)


## Compute stable steps + repeat datasets (prevents “ran out of data”)

In [ ]:
cat_n = len(glob.glob(os.path.join(BASE, "Cat", "*.jpg")))
dog_n = len(glob.glob(os.path.join(BASE, "Dog", "*.jpg")))
total = cat_n + dog_n

train_count = int(total * 0.8)
val_count = total - train_count

steps_per_epoch = train_count // BATCH_SIZE
val_steps = val_count // BATCH_SIZE

print("Total:", total, "Train approx:", train_count, "Val approx:", val_count)
print("steps_per_epoch:", steps_per_epoch, "val_steps:", val_steps)

train_ds_r = train_ds.repeat()
val_ds_r   = val_ds.repeat()


## Cell 6 — Data augmentation layer

In [ ]:
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
])


## Build model (EfficientNetB0 transfer learning)

In [ ]:
base = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,)
)
base.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_aug(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)

# IMPORTANT: float32 output for stability with mixed precision
outputs = tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

model.summary()


## Callbacks (best practice)

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/cats_vs_dogs_best.keras",
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]


## Train baseline (5 epochs)

In [ ]:
history = model.fit(
    train_ds_r,
    validation_data=val_ds_r,
    epochs=5,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
    callbacks=callbacks,
    verbose=1
)


## Fine-tune last 20 layers

In [ ]:
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

history_ft = model.fit(
    train_ds_r,
    validation_data=val_ds_r,
    epochs=3,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
    callbacks=callbacks,
    verbose=1
)


## Plot curves

In [ ]:
import matplotlib.pyplot as plt

def plot_hist(h, title):
    plt.figure()
    plt.plot(h.history["accuracy"], label="train_acc")
    plt.plot(h.history["val_accuracy"], label="val_acc")
    plt.title(f"{title} - Accuracy")
    plt.legend()
    plt.show()

    plt.figure()
    plt.plot(h.history["auc"], label="train_auc")
    plt.plot(h.history["val_auc"], label="val_auc")
    plt.title(f"{title} - AUC")
    plt.legend()
    plt.show()

plot_hist(history, "Baseline")
plot_hist(history_ft, "Fine-tune")


## Proper evaluation (confusion matrix + report)

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

y_true, y_prob = [], []

for images, labels in val_ds.take(val_steps):
    probs = model.predict(images, verbose=0).ravel()
    y_true.append(labels.numpy())
    y_prob.append(probs)

y_true = np.concatenate(y_true)
y_prob = np.concatenate(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=class_names))
print("ROC AUC:", roc_auc_score(y_true, y_prob))


## “Business choice” threshold

In [ ]:
from sklearn.metrics import precision_score, recall_score
import numpy as np

thresholds = np.arange(0.05, 0.96, 0.01)

rows = []
for t in thresholds:
    pred = (y_prob >= t).astype(int)
    rows.append({
        "t": float(t),
        "acc": float((pred == y_true).mean()),
        "precision_dog": float(precision_score(y_true, pred)),
        "recall_dog": float(recall_score(y_true, pred)),
    })

# Example constraints
# Choose highest precision while keeping recall >= 0.95
candidates_precision = [r for r in rows if r["recall_dog"] >= 0.95]
best_precision = max(candidates_precision, key=lambda r: r["precision_dog"]) if candidates_precision else max(rows, key=lambda r: r["precision_dog"])

# Choose highest recall while keeping precision >= 0.95
candidates_recall = [r for r in rows if r["precision_dog"] >= 0.95]
best_recall = max(candidates_recall, key=lambda r: r["recall_dog"]) if candidates_recall else max(rows, key=lambda r: r["recall_dog"])

print("Best precision(Dog) with recall>=0.95:", best_precision)
print("Best recall(Dog) with precision>=0.95:", best_recall)


## Misclassified image gallery

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Collect probabilities + true labels from validation
y_true, y_prob, x_images = [], [], []

for images, labels in val_ds.take(val_steps):
    probs = model.predict(images, verbose=0).ravel()
    y_true.append(labels.numpy())
    y_prob.append(probs)
    x_images.append(images.numpy().astype("uint8"))

y_true = np.concatenate(y_true)
y_prob = np.concatenate(y_prob)
x_images = np.concatenate(x_images, axis=0)

y_pred = (y_prob >= 0.5).astype(int)

mis_idx = np.where(y_pred != y_true)[0]
print("Misclassified:", len(mis_idx), "out of", len(y_true))

# Show up to N misclassified examples
N = min(12, len(mis_idx))
plt.figure(figsize=(14, 10))

for i in range(N):
    idx = mis_idx[i]
    ax = plt.subplot(3, 4, i + 1)
    plt.imshow(x_images[idx])
    true_name = class_names[int(y_true[idx])]
    pred_name = class_names[int(y_pred[idx])]
    plt.title(f"T:{true_name} / P:{pred_name}\nprob(dog)={y_prob[idx]:.2f}")
    plt.axis("off")

plt.tight_layout()
plt.show()


## Showing “most confident wrong”

In [ ]:
# confidence = how far prob is from the decision boundary 0.5
confidence = np.abs(y_prob - 0.5)
mis_idx = np.where(y_pred != y_true)[0]

# Show the most confident mistakes (often mislabeled or tricky images)
top_mis = mis_idx[np.argsort(-confidence[mis_idx])][:12]

plt.figure(figsize=(14, 10))
for i, idx in enumerate(top_mis):
    ax = plt.subplot(3, 4, i+1)
    plt.imshow(x_images[idx])
    true_name = class_names[int(y_true[idx])]
    pred_name = class_names[int(y_pred[idx])]
    plt.title(f"CONF WRONG\nT:{true_name} / P:{pred_name}\nprob(dog)={y_prob[idx]:.2f}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## Threshold tuning (finds the best threshold for F1 or accuracy)

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

thresholds = np.arange(0.05, 0.96, 0.01)

f1s = []
accs = []

for t in thresholds:
    pred_t = (y_prob >= t).astype(int)
    f1s.append(f1_score(y_true, pred_t))
    accs.append(accuracy_score(y_true, pred_t))

best_f1_idx = int(np.argmax(f1s))
best_acc_idx = int(np.argmax(accs))

best_f1_t = thresholds[best_f1_idx]
best_acc_t = thresholds[best_acc_idx]

print("Best F1 threshold:", best_f1_t, "F1:", f1s[best_f1_idx])
print("Best Acc threshold:", best_acc_t, "Acc:", accs[best_acc_idx])


## Plot threshold curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(thresholds, f1s, label="F1")
plt.plot(thresholds, accs, label="Accuracy")
plt.xlabel("Threshold (prob(dog) >= t)")
plt.ylabel("Score")
plt.title("Threshold tuning")
plt.legend()
plt.show()


## Reprint report using the best F1 threshold

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

t = best_f1_t
y_pred_best = (y_prob >= t).astype(int)

print("Using threshold:", t)
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred_best))
print("\nClassification Report:\n", classification_report(y_true, y_pred_best, target_names=class_names))


## Save the trained model (already checkpointed, but this is explicit)

In [ ]:
MODEL_PATH = "/kaggle/working/cats_vs_dogs_final.keras"
model.save(MODEL_PATH)
print("Saved to:", MODEL_PATH)

## Load it back

In [ ]:
loaded_model = tf.keras.models.load_model(MODEL_PATH)
print("Loaded model ok.")


## Single-image prediction helper

In [ ]:
import os, glob
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

def predict_image(model, image_path, img_size=(224, 224), threshold=0.5, class_names=None):
    """
    Predict Cat/Dog for a single image.

    Args:
        model: Trained/loaded tf.keras model.
        image_path: Path to the image file.
        img_size: (H, W) resize size.
        threshold: Decision threshold for prob(dog).
        class_names: List like ['Cat','Dog'] (index 0=Cat, 1=Dog).

    Returns:
        prob_dog (float), pred_label (str), img (PIL.Image)
    """
    if class_names is None:
        class_names = ["Cat", "Dog"]

    img = Image.open(image_path).convert("RGB").resize(img_size)
    x = np.array(img, dtype=np.float32)
    x = np.expand_dims(x, axis=0)  # (1, H, W, 3)

    prob_dog = float(model.predict(x, verbose=0).ravel()[0])
    pred_idx = 1 if prob_dog >= threshold else 0
    pred_label = class_names[pred_idx]

    return prob_dog, pred_label, img

# Example (pick any image file path)
example_path = glob.glob(os.path.join(BASE, "Cat", "*.jpg"))[0]

prob, label, img = predict_image(
    loaded_model,
    example_path,
    img_size=IMG_SIZE,
    threshold=float(best_f1_t),
    class_names=class_names
)

print("Path:", example_path)
print("prob(dog):", round(prob, 4), "-> predicted:", label)

plt.figure()
plt.imshow(img)
plt.axis("off")
plt.title(f"Pred: {label} | prob(dog)={prob:.2f} | thr={float(best_f1_t):.2f}")
plt.show()


## Show “most uncertain” images

In [ ]:
uncertainty = np.abs(y_prob - 0.5)
uncertain_idx = np.argsort(uncertainty)[:12]

plt.figure(figsize=(14, 10))
for i, idx in enumerate(uncertain_idx):
    ax = plt.subplot(3, 4, i+1)
    plt.imshow(x_images[idx])
    true_name = class_names[int(y_true[idx])]
    pred_name = class_names[int(y_prob[idx] >= 0.5)]
    plt.title(f"T:{true_name} / P:{pred_name}\nprob(dog)={y_prob[idx]:.2f}")
    plt.axis("off")
plt.tight_layout()
plt.show()


# Cats vs Dogs — Final Report
**Date:** 2025-12-25

## Dataset Stats
- **Dataset:** Kaggle Cats vs Dogs (PetImages)
- **Classes:** Cat, Dog
- **Cat images:** 12,490
- **Dog images:** 12,469
- **Total images:** 24,959
- **Train/Val split:** 80% / 20% (directory split w/ seed=42)
- **Image size:** 224 × 224
- **Batch size:** 32
- **Validation batches used:** 156 (≈ 4,991 images)

## Model Summary
- Backbone: EfficientNetB0 (ImageNet pre-trained), frozen for baseline; fine-tuned last 20 layers.
- Head: GlobalAveragePooling2D → Dropout(0.2) → Dense(1, sigmoid).
- Total params: 4,050,852  |  Trainable: 1,281  |  Non-trainable: 4,049,571

## Training Results
- Training strategy: Transfer learning baseline (Adam lr=1e-3) with early stopping on val AUC; then fine-tuning (Adam lr=1e-4) for last 20 backbone layers.
- Best checkpoint (by val AUC): epoch 2 of baseline training.
- Baseline (best epoch): val_accuracy ≈ 0.9924, val_auc ≈ 0.99939, val_loss ≈ 0.0251
- Fine-tune (best epoch): val_accuracy ≈ 0.9912, val_auc ≈ 0.9994, val_loss ≈ 0.0225

## Evaluation — Threshold = 0.50
- **ROC AUC:** 0.999529

**Confusion Matrix** (rows=true [Cat, Dog], cols=pred [Cat, Dog])
```text
[[2428   29]
 [  15 2519]]
```

**Classification Report**
```text
              precision    recall  f1-score   support

         Cat       0.99      0.99      0.99      2457
         Dog       0.99      0.99      0.99      2534

    accuracy                           0.99      4991
   macro avg       0.99      0.99      0.99      4991
weighted avg       0.99      0.99      0.99      4991

```

## Threshold Tuning
- Threshold sweep: t ∈ [0.05, 0.95] step 0.01 on prob(dog).
- Best threshold (F1 and Accuracy): t = 0.84
- At t=0.84: Accuracy ≈ 0.99319, F1 ≈ 0.99326
- Error trade-off: fewer Cat→Dog false positives (7) but more Dog→Cat false negatives (27).

## Evaluation — Best F1 Threshold = 0.84
**Confusion Matrix** (rows=true [Cat, Dog], cols=pred [Cat, Dog])
```text
[[2450    7]
 [  27 2507]]
```

**Classification Report**
```text
              precision    recall  f1-score   support

         Cat       0.99      1.00      0.99      2457
         Dog       1.00      0.99      0.99      2534

    accuracy                           0.99      4991
   macro avg       0.99      0.99      0.99      4991
weighted avg       0.99      0.99      0.99      4991

```

## Saved Model Paths
- `/kaggle/working/cats_vs_dogs_best.keras  (best by val_auc)`
- `/kaggle/working/cats_vs_dogs_final.keras (final saved model)`
